# ScratchGCN 과 학습 함수 직접 다시 짜기

2주차 PracticeCode_2 의 ScratchGCN(라이브러리 없이 torch 로 만든 GCN), 평균 모으기 GNN, 정확도 함수와 학습 함수를 한 줄씩 쳐 보고 직접 짜요. 클래스(c05, c06), nn.Module(c10), 학습 루프(c11)를 먼저 하면 좋아요. torch 가 필요해서 Colab 에서 열려요.

**하는 법**
1. 위 메뉴 **런타임 > 런타임 유형 변경** 은 CPU 그대로 두어도 돼요.
2. 문제마다 **내 코드 칸**을 채우고 실행(Shift+Enter)한 뒤, 바로 아래 **채점 칸**을 실행해요.
3. `통과!` 가 나오면 성공, 빨간 `AssertionError` 가 나오면 마지막 줄의 한국어 안내를 읽고 고쳐요.
4. 막히면 **힌트**를 한 단계씩 펼쳐 보고, 그래도 안 되면 **정답 보기**를 펼쳐요.

## 1. GCNLayer 클래스 치기  (따라 치기)

GCN 한 층을 클래스로 쳐요. 들어온 표현 H 에 가중치 W 를 곱하고, 정규화 인접행렬 A_hat 으로 이웃끼리 섞고, ReLU 로 음수를 지워요.

- 이 층 하나가 식 $H' = \mathrm{ReLU}(\hat{A} H W)$ 예요. $\hat{A}$ 는 w2-messagepass 에서 만든 정규화 인접행렬, H 는 노드마다 한 줄씩인 표현 표, W 는 학습으로 바뀌는 가중치예요.
- `class GCNLayer(nn.Module):` 는 PyTorch 가 준 기본 틀 nn.Module 을 물려받아요. `super().__init__()` 로 기본 틀 준비를 먼저 해요 (c06, c10).
- `self.W = nn.Linear(in_dim, out_dim, bias=False)` 는 가중치 표를 속성으로 달아요. `self.W(H)` 는 `H @ W.T` 를 계산해 줘요 (c10).
- `torch.relu(out) if activation else out` 는 한 줄 if 예요. activation 이 True 면 앞, False 면 뒤 값을 돌려줘요.

아래 코드를 **보면서 직접 쳐 보세요** (복사하지 말고요):

```python
import torch
import torch.nn as nn

class GCNLayer(nn.Module):
    '''One layer: H' = activation(A_hat H W).'''

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out
```

<details><summary>생각 순서 보기 (힌트보다 먼저)</summary>

1. 무엇을 기억하나부터 정해요. 층 하나가 들고 있을 건 가중치 표 하나뿐이라 `__init__` 에 `self.W` 만 만들어요.
2. 무엇을 하나도 정해요. `forward` 는 표현 표 `H` (노드 수, in_dim) 와 `A_hat` (노드 수, 노드 수) 를 받아 (노드 수, out_dim) 표를 돌려줘요.
3. 식 ReLU(A_hat H W) 를 안쪽부터 읽어요. 먼저 `self.W(H)` 로 칸 수를 바꾸고, 그다음 `A_hat @` 로 이웃끼리 섞어요.
4. 마지막 층은 음수 점수도 내야 해서, `activation` 이 False 면 ReLU 를 건너뛰어요.
5. 조심할 곳: `super().__init__()` 가 `self.W` 줄보다 먼저예요. 확인은 `GCNLayer(3, 2)` 에 (4, 3) 표를 넣어 결과 모양이 (4, 2) 인지 print 해요.

슈도코드

```text
GCNLayer 를 nn.Module 에서 물려받는다
__init__ 에서 in_dim, out_dim 을 받는다
    부모 준비를 먼저 한다
    in_dim 칸을 out_dim 칸으로 바꾸는 bias 없는 Linear 를 self.W 에 기억한다
forward 에서 H, A_hat, activation 을 받는다
    self.W 로 H 의 칸 수를 바꾼다
    그 왼쪽에 A_hat 을 행렬곱해 out 을 만든다
    activation 이 참이면 out 에 ReLU 를 씌워 돌려준다
    아니면 out 을 그대로 돌려준다
```

</details>

<details><summary>힌트 1</summary>

class 줄과 def 줄 끝의 콜론

</details>

<details><summary>힌트 2</summary>

super().__init__() 를 먼저

</details>

<details><summary>힌트 3</summary>

forward 는 A_hat @ self.W(H)

</details>

원본: PracticeCode_2.ipynb 셀 56


In [ ]:
# 여기에 코드를 쳐 보세요


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F
assert 'GCNLayer' in globals(), "GCNLayer 클래스를 만들어야 해요"
try:
    torch.manual_seed(0)
    _layer = GCNLayer(3, 2)
except Exception as _e:
    raise AssertionError(f"GCNLayer(3, 2) 로 층을 만들다가 에러가 났어요: {_e}. __init__(self, in_dim, out_dim) 과 super().__init__() 을 확인해요")
assert isinstance(_layer, nn.Module), "GCNLayer 은 nn.Module 을 물려받아야 해요: class GCNLayer(nn.Module):"
assert hasattr(_layer, 'W') and isinstance(_layer.W, nn.Linear), "self.W = nn.Linear(in_dim, out_dim, bias=False) 가 있어야 해요"
assert tuple(_layer.W.weight.shape) == (2, 3), f"W 의 weight 모양은 (출력 2, 입력 3) 이어야 해요. 지금은 {tuple(_layer.W.weight.shape)}"
assert _layer.W.bias is None, "원본은 bias=False 예요. 더하는 숫자(bias)가 없어야 해요"
torch.manual_seed(1)
_H = torch.randn(4, 3)
_Ah = torch.rand(4, 4)
try:
    _o1 = _layer(_H, _Ah)
    _o2 = _layer(_H, _Ah, activation=False)
except Exception as _e:
    raise AssertionError(f"층을 부르다가 에러가 났어요: {_e}. forward(self, H, A_hat, activation=True) 를 확인해요")
_raw = _Ah @ (_H @ _layer.W.weight.T)
assert _o2 is not None and torch.allclose(_o2, _raw, atol=1e-5), "activation=False 면 A_hat @ self.W(H) 그대로 돌려줘야 해요"
assert torch.allclose(_o1, torch.relu(_raw), atol=1e-5), "activation=True 면 torch.relu 로 음수를 0 으로 바꿔야 해요"

print("통과! 원본 셀 56 의 첫 클래스예요. PyG 의 GCNConv 도 속에서는 이 계산을 해요.")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn

class GCNLayer(nn.Module):
    '''One layer: H' = activation(A_hat H W).'''

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out
```

</details>

## 2. torch 로 A_hat 만들기  (빈칸 채우기)

8명짜리 그래프의 인접행렬 A 에서 자기 자신 고리를 더하고, 차수로 양쪽을 나눠 A_hat 을 만들어요. ScratchGCN 에 넣을 표예요.

- 원본은 PyG 의 가라테 클럽(34명)을 쓰지만, 채점을 빠르게 하려고 8명짜리 가짜 그래프(0~3번 한 무리, 4~7번 한 무리, 3번과 4번이 다리)로 바꿨어요. 내려받는 데이터는 없어요.
- 원본은 셀 30 의 `build_gcn_normalized_adjacency` (PyG `add_self_loops`, `degree` 사용)로 A_hat 을 만들어요. 여기서는 같은 식 $\hat{A} = \tilde{D}^{-1/2}(A + I)\tilde{D}^{-1/2}$ 를 행렬 연산으로 바로 써요 (c12 와 같은 방법).
- `torch.eye(n)` 은 대각선만 1 인 표, 즉 식의 I 예요. `A_tilde.sum(dim=1)` 은 줄마다(가로로) 더해서 차수를 구해요 (c09).
- `deg.pow(-0.5)` 는 1/루트(차수), `torch.diag(벡터)` 는 그 값을 대각선에 놓은 표예요. `@` 는 행렬곱이에요.

<details><summary>생각 순서 보기 (힌트보다 먼저)</summary>

1. 입력은 엣지 목록 `edges` 와 노드 수 8, 출력은 (8, 8) 표 `A_hat` 이에요.
2. 식 D^-1/2 (A + I) D^-1/2 를 계산 순서로 나눠요. 자기 고리 더하기, 차수 구하기, 1/루트(차수) 를 대각선에 놓기, 양쪽에서 곱하기예요.
3. 노드 1 만 손으로 풀어요. 이웃은 0 과 2, 자기 고리까지 차수는 3 이에요. 그래서 `A_hat[1, 1]` 은 1/루트3 x 1 x 1/루트3 = 1/3 이에요.
4. 조심할 곳: 차수는 자기 고리를 더한 `A_tilde` 로 구해요. 가운데에 곱하는 표도 `A` 가 아니라 `A_tilde` 예요.
5. 확인: `A_hat[1, 1]` 을 print 해서 0.3333 이 나오면 맞아요.

슈도코드

```text
8 x 8 크기 0 표 A 를 만든다
엣지 (u, v) 마다
    A[u, v] 와 A[v, u] 를 1 로 둔다
A 에 단위행렬을 더해 A_tilde 를 만든다
A_tilde 를 줄마다 더해 차수 deg 를 구한다
deg 의 -0.5 제곱을 대각선에 놓아 D_inv_sqrt 를 만든다
D_inv_sqrt, A_tilde, D_inv_sqrt 를 차례로 행렬곱해 A_hat 을 얻는다
```

</details>

<details><summary>힌트 1</summary>

I 는 torch.eye(num_nodes)

</details>

<details><summary>힌트 2</summary>

차수는 가로 합 dim=1, 1/루트는 pow(-0.5)

</details>

<details><summary>힌트 3</summary>

마지막 줄은 D_inv_sqrt @ A_tilde @ D_inv_sqrt

</details>

원본: PracticeCode_2.ipynb 셀 30, 53


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

edges = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
num_nodes = 8
A = torch.zeros(num_nodes, num_nodes)
for u, v in edges:
    A[u, v] = 1.0
    A[v, u] = 1.0
A_tilde = A + torch.___(num_nodes)
deg = A_tilde.sum(dim=___)
D_inv_sqrt = torch.diag(deg.pow(___))
A_hat = D_inv_sqrt @ ___ @ D_inv_sqrt

print(A_hat.sum(dim=1))


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F

class _RGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out

class _RScratchGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = _RGCNLayer(in_dim, hidden_dim)
        self.conv2 = _RGCNLayer(hidden_dim, num_classes)

    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)

def _r_graph():
    _e = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
    _A = torch.zeros(8, 8)
    for _u, _v in _e:
        _A[_u, _v] = 1.0
        _A[_v, _u] = 1.0
    _At = _A + torch.eye(8)
    _Dm = torch.diag(_At.sum(dim=1).pow(-0.5))
    _m = torch.zeros(8, dtype=torch.bool)
    _m[0] = True
    _m[7] = True
    return _A, _Dm @ _At @ _Dm, torch.eye(8), torch.tensor([0, 0, 0, 0, 1, 1, 1, 1]), _m

def _r_train(model, x, support, y, train_mask, epochs, lr):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    hl, ht, hf = [], [], []
    for _ep in range(1, epochs + 1):
        model.train()
        opt.zero_grad()
        lg = model(x, support)
        l = F.cross_entropy(lg[train_mask], y[train_mask])
        l.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            p = model(x, support).argmax(dim=1)
            ht.append((p[train_mask] == y[train_mask]).float().mean().item())
            hf.append((p == y).float().mean().item())
        hl.append(l.item())
    return hl, ht, hf

_A8, _AH8, _X8, _Y8, _M8 = _r_graph()
assert 'A_hat' in globals() and tuple(A_hat.shape) == (8, 8), "A_hat 은 8 x 8 이어야 해요"
assert 'deg' in globals() and torch.allclose(deg, _A8.sum(dim=1) + 1), f"deg 는 자기 자신 고리까지 센 차수 {(_A8.sum(dim=1) + 1).tolist()} 여야 해요. sum 의 dim=1 과 torch.eye 를 확인해요"
assert torch.allclose(A_hat, _AH8, atol=1e-6), "A_hat 값이 식과 달라요. D_inv_sqrt @ A_tilde @ D_inv_sqrt 순서와 pow(-0.5) 를 확인해요"

print("통과! 이 A_hat 을 뒤 실습에서 계속 모델에 넣어요.")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn
import torch.nn.functional as F

edges = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
num_nodes = 8
A = torch.zeros(num_nodes, num_nodes)
for u, v in edges:
    A[u, v] = 1.0
    A[v, u] = 1.0
A_tilde = A + torch.eye(num_nodes)
deg = A_tilde.sum(dim=1)
D_inv_sqrt = torch.diag(deg.pow(-0.5))
A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt

print(A_hat.sum(dim=1))

```

</details>

## 3. 층을 만들자마자 에러가 나요  (고치기)

GCNLayer(3, 2) 를 만들면 `cannot assign module before Module.__init__() call` 에러가 나요. 빠진 한 줄을 넣어요.

- 에러 메시지 마지막 줄을 읽어요: "Module.__init__() 부르기 전에 모듈을 달 수 없다" 는 뜻이에요.
- nn.Module 을 물려받으면 `__init__` 첫 줄에서 부모 준비 `super().__init__()` 를 꼭 해야 해요. 그래야 self.W 같은 층을 속성으로 달 수 있어요 (c06, c10-05).

<details><summary>생각 순서 보기 (힌트보다 먼저)</summary>

1. 에러 메시지 마지막 줄부터 읽어요. `cannot assign module before Module.__init__() call` 은 부모 준비를 하기 전에 층을 달았다는 말이에요.
2. 몇 번째 줄인지 찾아요. 트레이스백이 가리키는 줄은 `__init__` 안의 `self.W = nn.Linear(...)` 예요.
3. 그 줄 앞에 무엇이 있어야 하나 떠올려요. nn.Module 을 물려받은 클래스는 `__init__` 첫 줄이 `super().__init__()` 예요.
4. 확인: 고친 뒤 `print(layer)` 결과에 `(W): Linear(in_features=3, out_features=2, bias=False)` 가 보이면 맞아요.

슈도코드

```text
에러 마지막 줄을 읽는다
트레이스백에서 문제가 난 self.W 줄을 찾는다
__init__ 첫 줄에 부모 준비가 있는지 본다
    없으면 self.W 줄 위에 부모 준비 한 줄을 넣는다
다시 실행해 layer 를 print 한 결과에 W 가 보이는지 본다
```

</details>

<details><summary>힌트 1</summary>

에러는 self.W = ... 줄에서 나요

</details>

<details><summary>힌트 2</summary>

그 줄 위에 부모 틀 준비가 없어요

</details>

<details><summary>힌트 3</summary>

def __init__ 바로 아래에 super().__init__()

</details>

원본: PracticeCode_2.ipynb 셀 56


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GCNLayer(nn.Module):
    '''One layer: H' = activation(A_hat H W).'''

    def __init__(self, in_dim, out_dim):
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out


layer = GCNLayer(3, 2)
print(layer)

In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F
assert 'GCNLayer' in globals(), "GCNLayer 클래스를 만들어야 해요"
try:
    torch.manual_seed(0)
    _layer = GCNLayer(3, 2)
except Exception as _e:
    raise AssertionError(f"GCNLayer(3, 2) 로 층을 만들다가 에러가 났어요: {_e}. __init__(self, in_dim, out_dim) 과 super().__init__() 을 확인해요")
assert isinstance(_layer, nn.Module), "GCNLayer 은 nn.Module 을 물려받아야 해요: class GCNLayer(nn.Module):"
assert hasattr(_layer, 'W') and isinstance(_layer.W, nn.Linear), "self.W = nn.Linear(in_dim, out_dim, bias=False) 가 있어야 해요"
assert tuple(_layer.W.weight.shape) == (2, 3), f"W 의 weight 모양은 (출력 2, 입력 3) 이어야 해요. 지금은 {tuple(_layer.W.weight.shape)}"
assert _layer.W.bias is None, "원본은 bias=False 예요. 더하는 숫자(bias)가 없어야 해요"
torch.manual_seed(1)
_H = torch.randn(4, 3)
_Ah = torch.rand(4, 4)
try:
    _o1 = _layer(_H, _Ah)
    _o2 = _layer(_H, _Ah, activation=False)
except Exception as _e:
    raise AssertionError(f"층을 부르다가 에러가 났어요: {_e}. forward(self, H, A_hat, activation=True) 를 확인해요")
_raw = _Ah @ (_H @ _layer.W.weight.T)
assert _o2 is not None and torch.allclose(_o2, _raw, atol=1e-5), "activation=False 면 A_hat @ self.W(H) 그대로 돌려줘야 해요"
assert torch.allclose(_o1, torch.relu(_raw), atol=1e-5), "activation=True 면 torch.relu 로 음수를 0 으로 바꿔야 해요"

print("통과! 노트북의 모든 모델 클래스(GCNLayer, ScratchGCN, BasicGNN)가 이 줄로 시작해요.")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn
import torch.nn.functional as F

class GCNLayer(nn.Module):
    '''One layer: H' = activation(A_hat H W).'''

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out


layer = GCNLayer(3, 2)
print(layer)
```

</details>

## 4. ScratchGCN 두 층 모델  (빈칸 채우기)

GCNLayer 두 개를 쌓아 모델을 만들어요. 첫 층은 입력 칸 수를 hidden_dim 으로, 둘째 층은 hidden_dim 을 반 수로 바꿔요.

- 원본은 PyG 의 가라테 클럽(34명)을 쓰지만, 채점을 빠르게 하려고 8명짜리 가짜 그래프(0~3번 한 무리, 4~7번 한 무리, 3번과 4번이 다리)로 바꿨어요. 내려받는 데이터는 없어요.
- 층 안에 층을 속성으로 달아요: `self.conv1 = GCNLayer(...)`. 붕어빵 틀 안에 작은 틀 두 개를 넣은 셈이에요 (c10).
- 첫 층 출력 칸 수와 둘째 층 입력 칸 수가 같아야 행렬곱이 돼요. 그래서 둘 다 hidden_dim 이에요.
- `return_embeddings=True` 면 첫 층 결과 h (숨은 표현)를 바로 돌려줘요. 원본은 이걸 t-SNE 그림(셀 57)에 써요.
- 마지막 층은 `activation=False` 예요. 반마다 점수(logits)를 그대로 내야 cross_entropy 에 넣을 수 있어요 (c11).

<details><summary>생각 순서 보기 (힌트보다 먼저)</summary>

1. 무엇을 기억하나: 층 두 개예요. `__init__` 에서 `self.conv1`, `self.conv2` 를 속성으로 만들어요.
2. 칸 수를 줄로 적어요. 입력 8칸이 conv1 을 지나 16칸, conv2 를 지나 2칸이 돼요. 그래서 conv1 출력과 conv2 입력이 둘 다 `hidden_dim` 이에요.
3. 무엇을 하나: `forward` 는 x 를 conv1 에 넣어 h 를 만들고, h 를 conv2 에 넣어 반별 점수를 내요.
4. `return_embeddings` 가 True 면 conv2 까지 가지 않고 h 를 바로 돌려줘요.
5. 조심할 곳: 마지막 층은 `activation=False` 예요. 점수는 음수도 될 수 있어야 해요. 확인은 `logits.shape` 가 `torch.Size([8, 2])` 인지 봐요.

슈도코드

```text
__init__ 에서 in_dim, hidden_dim, num_classes 를 받는다
    부모 준비를 한다
    in_dim 을 hidden_dim 으로 바꾸는 GCNLayer 를 self.conv1 에 기억한다
    hidden_dim 을 num_classes 로 바꾸는 GCNLayer 를 self.conv2 에 기억한다
forward 에서 x, A_hat, return_embeddings 를 받는다
    conv1 에 x 와 A_hat 을 넣고 ReLU 를 켜서 h 를 만든다
    return_embeddings 가 참이면 h 를 돌려준다
    아니면 conv2 에 h 와 A_hat 을 넣고 ReLU 를 끈 점수를 돌려준다
```

</details>

<details><summary>힌트 1</summary>

conv1 의 출력은 hidden_dim

</details>

<details><summary>힌트 2</summary>

conv2 의 입력도 hidden_dim

</details>

<details><summary>힌트 3</summary>

return_embeddings 면 h, 마지막 층은 activation=False

</details>

원본: PracticeCode_2.ipynb 셀 56


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GCNLayer(nn.Module):
    '''One layer: H' = activation(A_hat H W).'''

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out


class ScratchGCN(nn.Module):
    '''Two-layer GCN.'''

    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = GCNLayer(in_dim, ___)
        self.conv2 = GCNLayer(___, num_classes)

    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return ___
        return self.conv2(h, A_hat, activation=___)


edges = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
num_nodes = 8
A = torch.zeros(num_nodes, num_nodes)
for u, v in edges:
    A[u, v] = 1.0
    A[v, u] = 1.0
A_tilde = A + torch.eye(num_nodes)
deg = A_tilde.sum(dim=1)
D_inv_sqrt = torch.diag(deg.pow(-0.5))
A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt
x = torch.eye(num_nodes)
y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1])
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
train_mask[0] = True
train_mask[7] = True

torch.manual_seed(42)
scratch_gcn = ScratchGCN(x.size(1), hidden_dim=16, num_classes=2)
logits = scratch_gcn(x, A_hat)
print(logits.shape)


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F

class _RGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out

class _RScratchGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = _RGCNLayer(in_dim, hidden_dim)
        self.conv2 = _RGCNLayer(hidden_dim, num_classes)

    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)

def _r_graph():
    _e = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
    _A = torch.zeros(8, 8)
    for _u, _v in _e:
        _A[_u, _v] = 1.0
        _A[_v, _u] = 1.0
    _At = _A + torch.eye(8)
    _Dm = torch.diag(_At.sum(dim=1).pow(-0.5))
    _m = torch.zeros(8, dtype=torch.bool)
    _m[0] = True
    _m[7] = True
    return _A, _Dm @ _At @ _Dm, torch.eye(8), torch.tensor([0, 0, 0, 0, 1, 1, 1, 1]), _m

def _r_train(model, x, support, y, train_mask, epochs, lr):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    hl, ht, hf = [], [], []
    for _ep in range(1, epochs + 1):
        model.train()
        opt.zero_grad()
        lg = model(x, support)
        l = F.cross_entropy(lg[train_mask], y[train_mask])
        l.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            p = model(x, support).argmax(dim=1)
            ht.append((p[train_mask] == y[train_mask]).float().mean().item())
            hf.append((p == y).float().mean().item())
        hl.append(l.item())
    return hl, ht, hf

_A8, _AH8, _X8, _Y8, _M8 = _r_graph()
assert 'ScratchGCN' in globals(), "ScratchGCN 클래스를 만들어야 해요"
for _i, _h, _c in [(8, 16, 2), (5, 4, 3)]:
    try:
        torch.manual_seed(42)
        _m = ScratchGCN(_i, _h, _c)
    except Exception as _e:
        raise AssertionError(f"ScratchGCN({_i}, {_h}, {_c}) 를 만들다가 에러가 났어요: {_e}")
    torch.manual_seed(42)
    _r = _RScratchGCN(_i, _h, _c)
    assert hasattr(_m, 'conv1') and hasattr(_m, 'conv2'), "self.conv1, self.conv2 두 층이 있어야 해요"
    assert tuple(_m.conv1.W.weight.shape) == (_h, _i), f"conv1 은 GCNLayer(in_dim, hidden_dim) 이어야 해요. weight 모양이 {tuple(_m.conv1.W.weight.shape)} 예요"
    assert tuple(_m.conv2.W.weight.shape) == (_c, _h), f"conv2 는 GCNLayer(hidden_dim, num_classes) 이어야 해요. weight 모양이 {tuple(_m.conv2.W.weight.shape)} 예요"
    torch.manual_seed(3)
    _xx = torch.randn(_i, _i) if _i != 8 else _X8
    _aa = torch.rand(_i, _i) if _i != 8 else _AH8
    try:
        _out = _m(_xx, _aa)
        _emb = _m(_xx, _aa, return_embeddings=True)
    except Exception as _e:
        raise AssertionError(f"모델을 부르다가 에러가 났어요: {_e}. 층의 입력, 출력 칸 수를 확인해요")
    assert _out is not None and tuple(_out.shape) == (_i, _c), f"model(x, A_hat) 는 (노드 수 {_i}, 반 수 {_c}) 모양이어야 해요"
    assert _emb is not None and tuple(_emb.shape) == (_i, _h), "return_embeddings=True 면 첫 층 결과 h 를 돌려줘야 해요"
    assert torch.allclose(_out, _r(_xx, _aa), atol=1e-5), "마지막 층 결과가 원본과 달라요. 마지막 층은 activation=False 예요"
    assert torch.allclose(_emb, _r(_xx, _aa, return_embeddings=True), atol=1e-5), "숨은 표현 h 가 원본과 달라요. 첫 층은 activation=True 예요"
assert 'logits' in globals() and tuple(logits.shape) == (8, 2), "logits 는 (8, 2) 모양이어야 해요"

print("통과! 원본 셀 56 에서는 입력 34칸(원-핫), 숨은 칸 16, 반 2 로 만들어요.")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn
import torch.nn.functional as F

class GCNLayer(nn.Module):
    '''One layer: H' = activation(A_hat H W).'''

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out


class ScratchGCN(nn.Module):
    '''Two-layer GCN.'''

    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = GCNLayer(in_dim, hidden_dim)
        self.conv2 = GCNLayer(hidden_dim, num_classes)

    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)


edges = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
num_nodes = 8
A = torch.zeros(num_nodes, num_nodes)
for u, v in edges:
    A[u, v] = 1.0
    A[v, u] = 1.0
A_tilde = A + torch.eye(num_nodes)
deg = A_tilde.sum(dim=1)
D_inv_sqrt = torch.diag(deg.pow(-0.5))
A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt
x = torch.eye(num_nodes)
y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1])
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
train_mask[0] = True
train_mask[7] = True

torch.manual_seed(42)
scratch_gcn = ScratchGCN(x.size(1), hidden_dim=16, num_classes=2)
logits = scratch_gcn(x, A_hat)
print(logits.shape)

```

</details>

## 5. 행렬곱 모양이 안 맞아요  (고치기)

`mat1 and mat2 shapes cannot be multiplied (8x16 and 8x2)` 에러가 나요. 층 하나의 입력 칸 수를 고쳐요.

- 에러는 "8 x 16 표와 8 x 2 가중치를 곱할 수 없다" 는 뜻이에요. 행렬곱은 앞 표의 칸 수와 뒤 표의 줄 수가 같아야 해요 (c07, c09).
- 첫 층을 지나면 노드마다 16칸이 돼요. 그러니 둘째 층은 16칸을 받아야 해요.
- 에러가 나는 건 정상이에요. 메시지 마지막 줄의 숫자 두 쌍을 보고 어느 층이 어긋났는지 찾아요.

<details><summary>생각 순서 보기 (힌트보다 먼저)</summary>

1. 에러 마지막 줄의 숫자를 읽어요. `8x16 and 8x2` 는 16칸짜리 표에 8칸을 받는 가중치를 곱하려 했다는 뜻이에요.
2. 어디서 났나 나눠요. 16칸 표는 conv1 을 지난 h 뿐이니, h 를 받는 conv2 에서 났어요.
3. 그 층을 만든 줄을 봐요. `GCNLayer(in_dim, num_classes)` 라서 8칸을 받게 되어 있어요.
4. conv2 입력 칸 수는 conv1 출력 칸 수와 같은 `hidden_dim` 이어야 해요.
5. 확인: 헷갈리면 forward 안에서 `h.shape` 를 print 해요. (8, 16) 이 나와야 하고, 고친 뒤 `logits.shape` 는 (8, 2) 예요.

슈도코드

```text
에러 마지막 줄에서 두 모양 8x16 과 8x2 를 읽는다
16칸 표가 어느 층을 지난 값인지 찾는다
    conv1 을 지난 h 이므로 문제는 conv2 이다
conv2 를 만드는 줄의 입력 칸 수를 본다
    in_dim 을 hidden_dim 으로 바꾼다
다시 실행해 logits 모양이 8 x 2 인지 본다
```

</details>

<details><summary>힌트 1</summary>

둘째 층을 만드는 줄을 봐요

</details>

<details><summary>힌트 2</summary>

둘째 층이 받는 것은 첫 층의 출력이에요

</details>

<details><summary>힌트 3</summary>

GCNLayer(hidden_dim, num_classes)

</details>

원본: PracticeCode_2.ipynb 셀 56


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GCNLayer(nn.Module):
    '''One layer: H' = activation(A_hat H W).'''

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out


class ScratchGCN(nn.Module):
    '''Two-layer GCN.'''

    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = GCNLayer(in_dim, hidden_dim)
        self.conv2 = GCNLayer(in_dim, num_classes)

    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)


edges = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
num_nodes = 8
A = torch.zeros(num_nodes, num_nodes)
for u, v in edges:
    A[u, v] = 1.0
    A[v, u] = 1.0
A_tilde = A + torch.eye(num_nodes)
deg = A_tilde.sum(dim=1)
D_inv_sqrt = torch.diag(deg.pow(-0.5))
A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt
x = torch.eye(num_nodes)
y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1])
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
train_mask[0] = True
train_mask[7] = True

torch.manual_seed(42)
scratch_gcn = ScratchGCN(x.size(1), hidden_dim=16, num_classes=2)
logits = scratch_gcn(x, A_hat)
print(logits.shape)


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F

class _RGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out

class _RScratchGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = _RGCNLayer(in_dim, hidden_dim)
        self.conv2 = _RGCNLayer(hidden_dim, num_classes)

    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)

def _r_graph():
    _e = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
    _A = torch.zeros(8, 8)
    for _u, _v in _e:
        _A[_u, _v] = 1.0
        _A[_v, _u] = 1.0
    _At = _A + torch.eye(8)
    _Dm = torch.diag(_At.sum(dim=1).pow(-0.5))
    _m = torch.zeros(8, dtype=torch.bool)
    _m[0] = True
    _m[7] = True
    return _A, _Dm @ _At @ _Dm, torch.eye(8), torch.tensor([0, 0, 0, 0, 1, 1, 1, 1]), _m

def _r_train(model, x, support, y, train_mask, epochs, lr):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    hl, ht, hf = [], [], []
    for _ep in range(1, epochs + 1):
        model.train()
        opt.zero_grad()
        lg = model(x, support)
        l = F.cross_entropy(lg[train_mask], y[train_mask])
        l.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            p = model(x, support).argmax(dim=1)
            ht.append((p[train_mask] == y[train_mask]).float().mean().item())
            hf.append((p == y).float().mean().item())
        hl.append(l.item())
    return hl, ht, hf

_A8, _AH8, _X8, _Y8, _M8 = _r_graph()
assert 'ScratchGCN' in globals(), "ScratchGCN 클래스를 만들어야 해요"
for _i, _h, _c in [(8, 16, 2), (5, 4, 3)]:
    try:
        torch.manual_seed(42)
        _m = ScratchGCN(_i, _h, _c)
    except Exception as _e:
        raise AssertionError(f"ScratchGCN({_i}, {_h}, {_c}) 를 만들다가 에러가 났어요: {_e}")
    torch.manual_seed(42)
    _r = _RScratchGCN(_i, _h, _c)
    assert hasattr(_m, 'conv1') and hasattr(_m, 'conv2'), "self.conv1, self.conv2 두 층이 있어야 해요"
    assert tuple(_m.conv1.W.weight.shape) == (_h, _i), f"conv1 은 GCNLayer(in_dim, hidden_dim) 이어야 해요. weight 모양이 {tuple(_m.conv1.W.weight.shape)} 예요"
    assert tuple(_m.conv2.W.weight.shape) == (_c, _h), f"conv2 는 GCNLayer(hidden_dim, num_classes) 이어야 해요. weight 모양이 {tuple(_m.conv2.W.weight.shape)} 예요"
    torch.manual_seed(3)
    _xx = torch.randn(_i, _i) if _i != 8 else _X8
    _aa = torch.rand(_i, _i) if _i != 8 else _AH8
    try:
        _out = _m(_xx, _aa)
        _emb = _m(_xx, _aa, return_embeddings=True)
    except Exception as _e:
        raise AssertionError(f"모델을 부르다가 에러가 났어요: {_e}. 층의 입력, 출력 칸 수를 확인해요")
    assert _out is not None and tuple(_out.shape) == (_i, _c), f"model(x, A_hat) 는 (노드 수 {_i}, 반 수 {_c}) 모양이어야 해요"
    assert _emb is not None and tuple(_emb.shape) == (_i, _h), "return_embeddings=True 면 첫 층 결과 h 를 돌려줘야 해요"
    assert torch.allclose(_out, _r(_xx, _aa), atol=1e-5), "마지막 층 결과가 원본과 달라요. 마지막 층은 activation=False 예요"
    assert torch.allclose(_emb, _r(_xx, _aa, return_embeddings=True), atol=1e-5), "숨은 표현 h 가 원본과 달라요. 첫 층은 activation=True 예요"

print("통과! 층을 쌓을 때 칸 수 이어 붙이기 실수는 GNN 코드에서 가장 흔한 에러예요.")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn
import torch.nn.functional as F

class GCNLayer(nn.Module):
    '''One layer: H' = activation(A_hat H W).'''

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out


class ScratchGCN(nn.Module):
    '''Two-layer GCN.'''

    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = GCNLayer(in_dim, hidden_dim)
        self.conv2 = GCNLayer(hidden_dim, num_classes)

    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)


edges = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
num_nodes = 8
A = torch.zeros(num_nodes, num_nodes)
for u, v in edges:
    A[u, v] = 1.0
    A[v, u] = 1.0
A_tilde = A + torch.eye(num_nodes)
deg = A_tilde.sum(dim=1)
D_inv_sqrt = torch.diag(deg.pow(-0.5))
A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt
x = torch.eye(num_nodes)
y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1])
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
train_mask[0] = True
train_mask[7] = True

torch.manual_seed(42)
scratch_gcn = ScratchGCN(x.size(1), hidden_dim=16, num_classes=2)
logits = scratch_gcn(x, A_hat)
print(logits.shape)

```

</details>

## 6. 점수에서 정확도 구하기  (따라 치기)

노드마다 반별 점수(logits)에서 가장 큰 반을 예측으로 고르고, 훈련 노드 정확도와 전체 정확도를 구해요.

- `logits.argmax(dim=1)` 은 줄마다 가장 큰 칸의 번호예요. 노드마다 예측 반이 나와요 (c11).
- `pred == y` 는 맞으면 True, 틀리면 False 인 텐서예요. `.float()` 로 1.0, 0.0 으로 바꾸고 `.mean()` 으로 평균, 즉 맞힌 비율을 내요.
- `pred[train_mask]` 는 train_mask 가 True 인 노드만 골라요. `.item()` 은 텐서에서 숫자 하나를 꺼내요 (c09).

아래 코드를 **보면서 직접 쳐 보세요** (복사하지 말고요):

```python
import torch
import torch.nn as nn

def accuracy_from_logits(logits, y, train_mask):
    pred = logits.argmax(dim=1)
    train_acc = (pred[train_mask] == y[train_mask]).float().mean().item()
    full_acc = (pred == y).float().mean().item()
    return pred, train_acc, full_acc

logits = torch.tensor([[2.0, 1.0], [0.1, 0.3], [1.5, -1.0], [0.0, 2.0]])
y = torch.tensor([0, 0, 0, 1])
train_mask = torch.tensor([True, False, False, True])
pred, train_acc, full_acc = accuracy_from_logits(logits, y, train_mask)
```

<details><summary>생각 순서 보기 (힌트보다 먼저)</summary>

1. 입력은 점수 표 `logits` (노드 수, 반 수), 정답 `y`, 고를 노드 표시 `train_mask` 예요. 출력은 예측, 훈련 정확도, 전체 정확도예요.
2. 예시를 손으로 풀어요. 점수 네 줄에서 줄마다 큰 칸은 0, 1, 0, 1 이에요. 정답 0, 0, 0, 1 과 비교하면 4개 중 3개가 맞아 전체 정확도는 0.75 예요.
3. 훈련 노드는 0 번과 3 번이고 둘 다 맞아서 훈련 정확도는 1.0 이에요.
4. 조심할 곳: 줄마다 골라야 해서 `argmax(dim=1)` 이에요. True/False 는 `.float()` 로 바꿔야 평균을 낼 수 있어요.

슈도코드

```text
logits 의 줄마다 가장 큰 칸 번호를 골라 pred 로 둔다
train_mask 가 참인 노드만 골라 pred 와 y 가 같은지 본다
같음은 1.0, 다름은 0.0 으로 바꿔 평균 내고 숫자로 꺼내 train_acc 로 둔다
모든 노드에서 같은 방법으로 full_acc 를 구한다
pred, train_acc, full_acc 를 돌려준다
```

</details>

<details><summary>힌트 1</summary>

argmax 는 dim=1 (줄마다)

</details>

<details><summary>힌트 2</summary>

비교 결과에 .float().mean().item()

</details>

<details><summary>힌트 3</summary>

return pred, train_acc, full_acc

</details>

원본: PracticeCode_2.ipynb 셀 49


In [ ]:
# 여기에 코드를 쳐 보세요


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F
assert 'accuracy_from_logits' in globals(), "accuracy_from_logits 함수를 만들어야 해요"
_res = accuracy_from_logits(torch.tensor([[2.0, 1.0], [0.1, 0.3], [1.5, -1.0], [0.0, 2.0]]), torch.tensor([0, 0, 0, 1]), torch.tensor([True, False, False, True]))
assert isinstance(_res, tuple) and len(_res) == 3, "pred, train_acc, full_acc 세 값을 돌려줘야 해요"
assert _res[0].tolist() == [0, 1, 0, 1], f"pred 는 [0, 1, 0, 1] 이어야 해요. argmax(dim=1) 을 확인해요"
assert abs(_res[1] - 1.0) < 1e-6 and abs(_res[2] - 0.75) < 1e-6, f"훈련 정확도 1.0, 전체 0.75 여야 해요. 지금은 {_res[1]}, {_res[2]}"
assert isinstance(_res[1], float), "정확도는 .item() 으로 꺼낸 숫자여야 해요"
_lg = torch.tensor([[0.0, 1.0, 5.0], [3.0, 0.0, 0.0], [0.0, 2.0, 1.0]])
_p, _t, _f = accuracy_from_logits(_lg, torch.tensor([2, 1, 1]), torch.tensor([False, True, True]))
assert _p.tolist() == [2, 0, 1] and abs(_t - 0.5) < 1e-6 and abs(_f - 2 / 3) < 1e-6, "반이 3개인 다른 예에서도 맞아야 해요"

print("통과! 원본 셀 49 첫 함수예요. 학습 중 바퀴마다 이 함수로 정확도를 기록해요.")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn

def accuracy_from_logits(logits, y, train_mask):
    pred = logits.argmax(dim=1)
    train_acc = (pred[train_mask] == y[train_mask]).float().mean().item()
    full_acc = (pred == y).float().mean().item()
    return pred, train_acc, full_acc

logits = torch.tensor([[2.0, 1.0], [0.1, 0.3], [1.5, -1.0], [0.0, 2.0]])
y = torch.tensor([0, 0, 0, 1])
train_mask = torch.tensor([True, False, False, True])
pred, train_acc, full_acc = accuracy_from_logits(logits, y, train_mask)
```

</details>

## 7. 학습 함수 train_dense_model  (빈칸 채우기)

ScratchGCN 을 60바퀴 학습하는 함수의 빈칸을 채워요. 바퀴마다 손실, 훈련 정확도, 전체 정확도를 기록해요.

- 원본은 PyG 의 가라테 클럽(34명)을 쓰지만, 채점을 빠르게 하려고 8명짜리 가짜 그래프(0~3번 한 무리, 4~7번 한 무리, 3번과 4번이 다리)로 바꿨어요. 내려받는 데이터는 없어요. 바퀴 수도 원본 300 대신 60 으로 줄였어요.
- 한 바퀴 = `model.train()` → `optimizer.zero_grad()` → 예측 → train_mask 노드만 손실 → `loss.backward()` → `optimizer.step()` 이에요 (c11 학습 루프).
- 그다음 `model.eval()` 과 `with torch.no_grad():` 안에서 다시 예측해 정확도를 재요. 채점할 때는 기울기 계산이 필요 없어요.
- `weight_decay=5e-4` 는 가중치가 너무 커지지 않게 살짝 누르는 설정이에요. 5e-4 는 0.0005 예요.
- `support` 는 A_hat 자리예요. 같은 함수로 BasicGNN(adj)도, ScratchGCN(A_hat)도 학습하려고 이름을 넓게 지었어요.

<details><summary>생각 순서 보기 (힌트보다 먼저)</summary>

1. 입력은 모델, 특징 `x`, 섞는 표 `support`, 정답 `y`, 훈련 노드 표시, 바퀴 수예요. 출력은 바퀴마다 적은 기록 리스트 세 개예요.
2. 반복 밖에서 한 번만 할 일과 반복 안에서 매번 할 일을 나눠요. 옵티마이저와 빈 리스트는 밖, 학습 한 걸음과 채점은 안이에요.
3. 학습 한 걸음은 늘 같은 순서예요. 기울기 지우기, 예측, 손실, backward, step 이에요.
4. 조심할 곳: 손실은 답을 아는 노드만 써야 해서 `logits[train_mask]` 와 `y[train_mask]` 를 같이 골라요.
5. 확인: 출력된 Epoch 줄에서 loss 가 점점 줄어들면 맞아요.

슈도코드

```text
Adam 옵티마이저와 빈 기록 리스트 세 개를 만든다
epoch 를 1 부터 epochs 까지 돌며
    학습 모드로 바꾸고 지난 기울기를 지운다
    model 에 x 와 support 를 넣어 logits 를 얻는다
    train_mask 노드만 골라 cross_entropy 손실을 구한다
    손실로 기울기를 구하고 가중치를 한 번 고친다
    평가 모드와 기울기 없이 다시 예측해 정확도 두 개를 잰다
    손실과 정확도 두 개를 리스트에 붙인다
리스트 세 개를 돌려준다
```

</details>

<details><summary>힌트 1</summary>

바퀴 시작에는 기울기 지우기

</details>

<details><summary>힌트 2</summary>

손실은 logits[train_mask] 와 y[train_mask]

</details>

<details><summary>힌트 3</summary>

backward 다음 step, 채점은 eval 과 no_grad

</details>

원본: PracticeCode_2.ipynb 셀 49, 56


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GCNLayer(nn.Module):
    '''One layer: H' = activation(A_hat H W).'''

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out


class ScratchGCN(nn.Module):
    '''Two-layer GCN.'''

    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = GCNLayer(in_dim, hidden_dim)
        self.conv2 = GCNLayer(hidden_dim, num_classes)

    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)


def accuracy_from_logits(logits, y, train_mask):
    pred = logits.argmax(dim=1)
    train_acc = (pred[train_mask] == y[train_mask]).float().mean().item()
    full_acc = (pred == y).float().mean().item()
    return pred, train_acc, full_acc


def train_dense_model(model, x, support, y, train_mask, epochs=300, lr=0.01, name="model"):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    history_loss, history_train, history_full = [], [], []

    print(f"Training {name}")
    print("=" * 60)
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.___()
        logits = model(x, support)
        loss = F.cross_entropy(logits[___], y[___])
        loss.___()
        optimizer.step()

        model.___()
        with torch.no_grad():
            logits_eval = model(x, support)
            _, train_acc, full_acc = accuracy_from_logits(logits_eval, y, train_mask)

        history_loss.append(loss.item())
        history_train.append(train_acc)
        history_full.append(full_acc)

        if epoch % 20 == 0 or epoch == 1:
            print(
                f"Epoch {epoch:03d}/{epochs} | loss={loss.item():.4f} "
                f"| train acc={train_acc:.4f} | full-graph acc={full_acc:.4f}"
            )

    print("=" * 60)
    return history_loss, history_train, history_full


edges = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
num_nodes = 8
A = torch.zeros(num_nodes, num_nodes)
for u, v in edges:
    A[u, v] = 1.0
    A[v, u] = 1.0
A_tilde = A + torch.eye(num_nodes)
deg = A_tilde.sum(dim=1)
D_inv_sqrt = torch.diag(deg.pow(-0.5))
A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt
x = torch.eye(num_nodes)
y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1])
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
train_mask[0] = True
train_mask[7] = True

torch.manual_seed(42)
scratch_gcn = ScratchGCN(x.size(1), hidden_dim=16, num_classes=2)
gcn_losses, gcn_train_accs, gcn_full_accs = train_dense_model(
    scratch_gcn, x, A_hat, y, train_mask, epochs=60, name="the scratch GCN"
)


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F

class _RGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out

class _RScratchGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = _RGCNLayer(in_dim, hidden_dim)
        self.conv2 = _RGCNLayer(hidden_dim, num_classes)

    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)

def _r_graph():
    _e = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
    _A = torch.zeros(8, 8)
    for _u, _v in _e:
        _A[_u, _v] = 1.0
        _A[_v, _u] = 1.0
    _At = _A + torch.eye(8)
    _Dm = torch.diag(_At.sum(dim=1).pow(-0.5))
    _m = torch.zeros(8, dtype=torch.bool)
    _m[0] = True
    _m[7] = True
    return _A, _Dm @ _At @ _Dm, torch.eye(8), torch.tensor([0, 0, 0, 0, 1, 1, 1, 1]), _m

def _r_train(model, x, support, y, train_mask, epochs, lr):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    hl, ht, hf = [], [], []
    for _ep in range(1, epochs + 1):
        model.train()
        opt.zero_grad()
        lg = model(x, support)
        l = F.cross_entropy(lg[train_mask], y[train_mask])
        l.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            p = model(x, support).argmax(dim=1)
            ht.append((p[train_mask] == y[train_mask]).float().mean().item())
            hf.append((p == y).float().mean().item())
        hl.append(l.item())
    return hl, ht, hf

_A8, _AH8, _X8, _Y8, _M8 = _r_graph()
assert 'gcn_losses' in globals() and len(gcn_losses) == 60, "gcn_losses 에 60바퀴 손실이 모여야 해요"
torch.manual_seed(42)
_rm = _RScratchGCN(8, 16, 2)
_rl, _rt, _rf = _r_train(_rm, _X8, _AH8, _Y8, _M8, 60, 0.01)
assert abs(gcn_losses[0] - _rl[0]) < 1e-5, f"첫 손실이 원본 {_rl[0]:.4f} 와 달라요(지금 {gcn_losses[0]:.4f}). train_mask 인 노드만 채점했는지 봐요"
assert max(abs(a - b) for a, b in zip(gcn_losses, _rl)) < 1e-4, "손실 기록이 원본과 달라요. zero_grad, backward, step 순서를 확인해요"
assert gcn_full_accs == _rf, "전체 정확도 기록이 원본과 달라요. model.eval() 과 torch.no_grad() 안에서 다시 계산했나요?"
assert gcn_losses[-1] < gcn_losses[0], "학습하면 손실이 줄어야 해요"

print("통과! 원본 셀 49 의 함수 그대로예요. 노트북은 이 함수 하나로 모델 두 개를 학습해요.")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn
import torch.nn.functional as F

class GCNLayer(nn.Module):
    '''One layer: H' = activation(A_hat H W).'''

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out


class ScratchGCN(nn.Module):
    '''Two-layer GCN.'''

    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = GCNLayer(in_dim, hidden_dim)
        self.conv2 = GCNLayer(hidden_dim, num_classes)

    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)


def accuracy_from_logits(logits, y, train_mask):
    pred = logits.argmax(dim=1)
    train_acc = (pred[train_mask] == y[train_mask]).float().mean().item()
    full_acc = (pred == y).float().mean().item()
    return pred, train_acc, full_acc


def train_dense_model(model, x, support, y, train_mask, epochs=300, lr=0.01, name="model"):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    history_loss, history_train, history_full = [], [], []

    print(f"Training {name}")
    print("=" * 60)
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(x, support)
        loss = F.cross_entropy(logits[train_mask], y[train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            logits_eval = model(x, support)
            _, train_acc, full_acc = accuracy_from_logits(logits_eval, y, train_mask)

        history_loss.append(loss.item())
        history_train.append(train_acc)
        history_full.append(full_acc)

        if epoch % 20 == 0 or epoch == 1:
            print(
                f"Epoch {epoch:03d}/{epochs} | loss={loss.item():.4f} "
                f"| train acc={train_acc:.4f} | full-graph acc={full_acc:.4f}"
            )

    print("=" * 60)
    return history_loss, history_train, history_full


edges = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
num_nodes = 8
A = torch.zeros(num_nodes, num_nodes)
for u, v in edges:
    A[u, v] = 1.0
    A[v, u] = 1.0
A_tilde = A + torch.eye(num_nodes)
deg = A_tilde.sum(dim=1)
D_inv_sqrt = torch.diag(deg.pow(-0.5))
A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt
x = torch.eye(num_nodes)
y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1])
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
train_mask[0] = True
train_mask[7] = True

torch.manual_seed(42)
scratch_gcn = ScratchGCN(x.size(1), hidden_dim=16, num_classes=2)
gcn_losses, gcn_train_accs, gcn_full_accs = train_dense_model(
    scratch_gcn, x, A_hat, y, train_mask, epochs=60, name="the scratch GCN"
)

```

</details>

## 8. 이웃 평균 한 번에 구하기  (따라 치기)

인접행렬 adj 와 특징 x 로 노드마다 이웃 특징의 평균을 한 번에 구해요. BasicGNN 의 모으기 방법이에요.

- `adj @ x` 의 줄 v 는 v 의 이웃 특징을 모두 더한 값이에요. 그걸 이웃 수로 나누면 평균이에요.
- `adj.sum(dim=1, keepdim=True)` 는 줄마다 더한 이웃 수를 (노드 수, 1) 세로 막대 모양으로 남겨요. keepdim 이 있어야 (노드 수, 칸 수) 표를 줄마다 나눌 수 있어요 (c07 브로드캐스팅).
- `.clamp(min=1.0)` 은 1 보다 작은 값을 1 로 올려요. 이웃이 없는 노드에서 0 으로 나누는 일을 막아요.
- 노드 3 은 아무와도 안 이어져 있어서 평균이 0 이에요.

아래 코드를 **보면서 직접 쳐 보세요** (복사하지 말고요):

```python
import torch

adj = torch.tensor([[0.0, 1.0, 1.0, 0.0], [1.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]])
x = torch.tensor([[1.0, 0.0], [0.0, 2.0], [4.0, 4.0], [9.0, 9.0]])
degree = adj.sum(dim=1, keepdim=True).clamp(min=1.0)
aggregated = (adj @ x) / degree
print(degree.shape)
print(aggregated)
```

<details><summary>생각 순서 보기 (힌트보다 먼저)</summary>

1. 입력은 인접행렬 `adj` (4, 4) 와 특징 `x` (4, 2), 출력은 노드마다 이웃 평균이 든 (4, 2) 표예요.
2. 노드 0 을 손으로 풀어요. 이웃은 1, 2 라서 합은 (0, 2) + (4, 4) = (4, 6), 이웃 수 2 로 나누면 (2, 3) 이에요.
3. 모든 노드의 이웃 합은 `adj @ x` 한 번이면 나와요. 이웃 수는 `adj` 를 줄마다 더한 값이에요.
4. 조심할 곳: 이웃 수는 `keepdim=True` 로 (4, 1) 모양을 지켜야 줄마다 나눠져요. 이웃이 없는 노드 3 은 `clamp(min=1.0)` 로 0 나누기를 막아요.
5. 확인: `aggregated` 첫 줄이 [2., 3.], 마지막 줄이 [0., 0.] 이면 맞아요.

슈도코드

```text
adj 를 줄마다 더해 노드별 이웃 수를 세로 막대 모양으로 남긴다
이웃 수가 1 보다 작으면 1 로 올린다
adj 와 x 를 행렬곱해 노드별 이웃 특징 합을 구한다
그 합을 이웃 수로 줄마다 나눠 aggregated 로 둔다
```

</details>

<details><summary>힌트 1</summary>

dim=1 은 가로(줄마다) 합

</details>

<details><summary>힌트 2</summary>

keepdim=True 와 clamp(min=1.0)

</details>

<details><summary>힌트 3</summary>

(adj @ x) / degree

</details>

원본: PracticeCode_2.ipynb 셀 48


In [ ]:
# 여기에 코드를 쳐 보세요


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
for _n in ['degree', 'aggregated']:
    assert _n in globals(), f"{_n} 를 만들어야 해요"
assert tuple(degree.shape) == (4, 1), f"degree 는 (4, 1) 모양이어야 해요. keepdim=True 를 확인해요. 지금은 {tuple(degree.shape)}"
assert degree.view(-1).tolist() == [2.0, 1.0, 1.0, 1.0], "degree 는 [2, 1, 1, 1] 이어야 해요. 노드 3 은 clamp 로 1 이 돼요"
assert torch.allclose(aggregated, torch.tensor([[2.0, 3.0], [1.0, 0.0], [1.0, 0.0], [0.0, 0.0]])), f"이웃 평균이 달라요. 지금은 {aggregated.tolist()}"

print("통과! 원본 셀 48 MeanAggregationLayer 의 forward 첫 두 줄이에요.")

<details><summary>정답 보기</summary>

```python
import torch

adj = torch.tensor([[0.0, 1.0, 1.0, 0.0], [1.0, 0.0, 0.0, 0.0], [1.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]])
x = torch.tensor([[1.0, 0.0], [0.0, 2.0], [4.0, 4.0], [9.0, 9.0]])
degree = adj.sum(dim=1, keepdim=True).clamp(min=1.0)
aggregated = (adj @ x) / degree
print(degree.shape)
print(aggregated)
```

</details>

## 9. 평균 모으기 GNN, BasicGNN  (빈칸 채우기)

내 특징(W_self)과 이웃 평균(W_neigh)을 따로 바꿔 더하는 층, 그리고 그 층 두 개와 분류기 한 개로 된 BasicGNN 을 완성해요.

- 원본은 PyG 의 가라테 클럽(34명)을 쓰지만, 채점을 빠르게 하려고 8명짜리 가짜 그래프(0~3번 한 무리, 4~7번 한 무리, 3번과 4번이 다리)로 바꿨어요. 내려받는 데이터는 없어요. 원본의 `to_dense_adj(data.edge_index, ...)[0]` 대신 직접 만든 인접행렬 A 를 adj 로 써요. 같은 0, 1 표예요.
- 층 식은 $h_v' = \mathrm{ReLU}(W_{self} h_v + W_{neigh} \cdot \text{이웃 평균})$ 이에요. GCN 과 달리 나와 이웃에 서로 다른 가중치를 써요.
- `self.classifier = nn.Linear(hidden_dim, num_classes)` 는 마지막에 반별 점수를 내는 층이에요.
- 08 에서 친 두 줄이 forward 안에 들어가요.

<details><summary>생각 순서 보기 (힌트보다 먼저)</summary>

1. 클래스가 두 개라 따로 생각해요. MeanAggregationLayer 가 기억할 것은 가중치 두 개, 내 몫 `W_self` 와 이웃 몫 `W_neigh` 예요.
2. MeanAggregationLayer 가 할 일: 08 의 이웃 평균을 구하고, 내 특징과 이웃 평균을 서로 다른 가중치로 바꿔 더한 뒤 ReLU 를 씌워요.
3. BasicGNN 이 기억할 것은 층 두 개와 분류기 하나예요. 칸 수는 8 에서 16, 16 에서 16, 분류기에서 16 에서 2 예요.
4. BasicGNN 이 할 일: layer1, layer2 를 차례로 지나 h 를 만들고, `return_embeddings` 면 h, 아니면 `self.classifier(h)` 를 돌려줘요.
5. 조심할 곳: `W_neigh` 에는 x 가 아니라 `aggregated` 를 넣어요. 확인은 `basic_logits.shape` 가 (8, 2), `basic_h.shape` 가 (8, 16) 인지 봐요.

슈도코드

```text
MeanAggregationLayer 의 __init__ 에서 W_self 와 W_neigh 를 bias 없이 기억한다
MeanAggregationLayer 의 forward 에서
    adj 를 줄마다 더해 이웃 수를 구하고 1 보다 작으면 1 로 올린다
    adj 와 x 의 행렬곱을 이웃 수로 나눠 aggregated 를 만든다
    W_self 로 바꾼 x 와 W_neigh 로 바꾼 aggregated 를 더해 ReLU 를 씌워 돌려준다
BasicGNN 의 __init__ 에서 layer1, layer2, classifier 를 기억한다
BasicGNN 의 forward 에서
    x 를 layer1, layer2 에 차례로 넣어 h 를 만든다
    return_embeddings 가 참이면 h 를 돌려준다
    아니면 classifier 에 h 를 넣은 점수를 돌려준다
```

</details>

<details><summary>힌트 1</summary>

이웃 수는 adj.sum(dim=1, keepdim=True)

</details>

<details><summary>힌트 2</summary>

평균은 (adj @ x) / degree, 더하는 것은 W_self(x) 와 W_neigh(aggregated)

</details>

<details><summary>힌트 3</summary>

layer2 입력은 hidden_dim, 마지막은 classifier(h)

</details>

원본: PracticeCode_2.ipynb 셀 48


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MeanAggregationLayer(nn.Module):
    '''Average neighbor messages, then combine with a linear map + ReLU.'''

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W_self = nn.Linear(in_dim, out_dim, bias=False)
        self.W_neigh = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, x, adj):
        degree = adj.sum(dim=1, keepdim=___).clamp(min=1.0)
        aggregated = (adj @ x) / ___
        return torch.relu(self.W_self(x) + self.W_neigh(___))


class BasicGNN(nn.Module):
    '''Two-layer mean-aggregation GNN for node classification.'''

    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.layer1 = MeanAggregationLayer(in_dim, hidden_dim)
        self.layer2 = MeanAggregationLayer(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, adj, return_embeddings=False):
        h = self.layer1(x, adj)
        h = self.layer2(h, adj)
        if return_embeddings:
            return h
        return self.___(h)


edges = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
num_nodes = 8
A = torch.zeros(num_nodes, num_nodes)
for u, v in edges:
    A[u, v] = 1.0
    A[v, u] = 1.0
x = torch.eye(num_nodes)
y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1])
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
train_mask[0] = True
train_mask[7] = True

adj = A
torch.manual_seed(42)
basic_gnn = BasicGNN(x.size(1), hidden_dim=16, num_classes=2)
basic_logits = basic_gnn(x, adj)
basic_h = basic_gnn(x, adj, return_embeddings=True)


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F

class _RGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out

class _RScratchGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = _RGCNLayer(in_dim, hidden_dim)
        self.conv2 = _RGCNLayer(hidden_dim, num_classes)

    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)

def _r_graph():
    _e = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
    _A = torch.zeros(8, 8)
    for _u, _v in _e:
        _A[_u, _v] = 1.0
        _A[_v, _u] = 1.0
    _At = _A + torch.eye(8)
    _Dm = torch.diag(_At.sum(dim=1).pow(-0.5))
    _m = torch.zeros(8, dtype=torch.bool)
    _m[0] = True
    _m[7] = True
    return _A, _Dm @ _At @ _Dm, torch.eye(8), torch.tensor([0, 0, 0, 0, 1, 1, 1, 1]), _m

def _r_train(model, x, support, y, train_mask, epochs, lr):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    hl, ht, hf = [], [], []
    for _ep in range(1, epochs + 1):
        model.train()
        opt.zero_grad()
        lg = model(x, support)
        l = F.cross_entropy(lg[train_mask], y[train_mask])
        l.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            p = model(x, support).argmax(dim=1)
            ht.append((p[train_mask] == y[train_mask]).float().mean().item())
            hf.append((p == y).float().mean().item())
        hl.append(l.item())
    return hl, ht, hf

_A8, _AH8, _X8, _Y8, _M8 = _r_graph()
class _RMean(nn.Module):
    def __init__(self, i, o):
        super().__init__()
        self.W_self = nn.Linear(i, o, bias=False)
        self.W_neigh = nn.Linear(i, o, bias=False)
    def forward(self, x, adj):
        d = adj.sum(dim=1, keepdim=True).clamp(min=1.0)
        return torch.relu(self.W_self(x) + self.W_neigh((adj @ x) / d))
class _RBasic(nn.Module):
    def __init__(self, i, h, c):
        super().__init__()
        self.layer1 = _RMean(i, h)
        self.layer2 = _RMean(h, h)
        self.classifier = nn.Linear(h, c)
    def forward(self, x, adj, return_embeddings=False):
        h = self.layer2(self.layer1(x, adj), adj)
        if return_embeddings:
            return h
        return self.classifier(h)
assert 'BasicGNN' in globals(), "BasicGNN 클래스가 있어야 해요"
torch.manual_seed(42)
_rb = _RBasic(8, 16, 2)
assert tuple(basic_logits.shape) == (8, 2) and tuple(basic_h.shape) == (8, 16), "logits 는 (8, 2), 숨은 표현은 (8, 16) 이어야 해요"
assert torch.allclose(basic_h, _rb(_X8, _A8, return_embeddings=True), atol=1e-5), "숨은 표현이 원본과 달라요. 이웃 평균의 keepdim, clamp 와 layer2 입력을 확인해요"
assert torch.allclose(basic_logits, _rb(_X8, _A8), atol=1e-5), "점수가 원본과 달라요. 마지막은 self.classifier(h) 예요"
torch.manual_seed(5)
_iso = torch.zeros(3, 3)
_iso[0, 1] = 1.0
_iso[1, 0] = 1.0
_xx = torch.randn(3, 4)
torch.manual_seed(6)
_m1 = MeanAggregationLayer(4, 2)
torch.manual_seed(6)
_m2 = _RMean(4, 2)
assert torch.allclose(_m1(_xx, _iso), _m2(_xx, _iso), atol=1e-5), "혼자인 노드가 있는 그래프에서 MeanAggregationLayer 결과가 달라요. clamp(min=1.0) 을 확인해요"

print("통과! 원본 셀 48 이에요. 셀 49 에서 이 모델도 같은 train_dense_model 로 학습해요.")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn
import torch.nn.functional as F

class MeanAggregationLayer(nn.Module):
    '''Average neighbor messages, then combine with a linear map + ReLU.'''

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W_self = nn.Linear(in_dim, out_dim, bias=False)
        self.W_neigh = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, x, adj):
        degree = adj.sum(dim=1, keepdim=True).clamp(min=1.0)
        aggregated = (adj @ x) / degree
        return torch.relu(self.W_self(x) + self.W_neigh(aggregated))


class BasicGNN(nn.Module):
    '''Two-layer mean-aggregation GNN for node classification.'''

    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.layer1 = MeanAggregationLayer(in_dim, hidden_dim)
        self.layer2 = MeanAggregationLayer(hidden_dim, hidden_dim)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, x, adj, return_embeddings=False):
        h = self.layer1(x, adj)
        h = self.layer2(h, adj)
        if return_embeddings:
            return h
        return self.classifier(h)


edges = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
num_nodes = 8
A = torch.zeros(num_nodes, num_nodes)
for u, v in edges:
    A[u, v] = 1.0
    A[v, u] = 1.0
x = torch.eye(num_nodes)
y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1])
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
train_mask[0] = True
train_mask[7] = True

adj = A
torch.manual_seed(42)
basic_gnn = BasicGNN(x.size(1), hidden_dim=16, num_classes=2)
basic_logits = basic_gnn(x, adj)
basic_h = basic_gnn(x, adj, return_embeddings=True)

```

</details>

## 10. GCNLayer 와 ScratchGCN 직접 짜기  (직접 짜기)

주석만 보고 GCNLayer 와 ScratchGCN 두 클래스를 짜요.

- 01(GCNLayer)과 04(ScratchGCN)를 보지 않고 다시 써 보는 거예요. 막히면 c10 의 nn.Module 틀부터 떠올려요.
- 채점은 서로 다른 크기 두 모델을 원본 코드와 같은 seed 로 만들어 결과가 똑같은지 봐요. 층을 만드는 순서(conv1 다음 conv2)가 같아야 가중치도 같아요.

<details><summary>생각 순서 보기 (힌트보다 먼저)</summary>

1. 클래스 두 개를 작은 것부터 짜요. GCNLayer 가 먼저 있어야 ScratchGCN 안에서 쓸 수 있어요.
2. GCNLayer. 무엇을 기억하나: 가중치 `self.W` 하나. 무엇을 하나: `A_hat @ self.W(H)` 를 만들고, 필요하면 ReLU 를 씌워요.
3. ScratchGCN. 무엇을 기억하나: `self.conv1` (in_dim 에서 hidden_dim), `self.conv2` (hidden_dim 에서 num_classes). 무엇을 하나: conv1 을 지나고, 필요하면 거기서 멈추고, 아니면 conv2 를 지나요.
4. 조심할 곳: 두 `__init__` 모두 첫 줄이 `super().__init__()` 예요. conv1 을 conv2 보다 먼저 만들어야 같은 seed 에서 가중치도 원본과 같아요.
5. 확인: `ScratchGCN(8, 16, 2)` 에 `torch.eye(8)` 과 8 x 8 표를 넣어 출력 모양이 (8, 2) 인지 print 해요.

슈도코드

```text
GCNLayer 를 nn.Module 에서 물려받는다
    __init__ 에서 부모 준비 뒤 bias 없는 Linear 를 self.W 로 기억한다
    forward 에서 self.W 를 거친 H 왼쪽에 A_hat 을 행렬곱해 out 을 만든다
    activation 이 참이면 out 에 ReLU 를, 아니면 out 을 돌려준다
ScratchGCN 을 nn.Module 에서 물려받는다
    __init__ 에서 부모 준비 뒤 conv1 을 만들고 그다음 conv2 를 만든다
    forward 에서 conv1 에 x 를 넣고 ReLU 를 켜서 h 를 만든다
    return_embeddings 가 참이면 h 를 돌려준다
    아니면 conv2 에 h 를 넣고 ReLU 를 끈 결과를 돌려준다
```

</details>

<details><summary>힌트 1</summary>

GCNLayer: __init__ 에 super().__init__() 와 self.W

</details>

<details><summary>힌트 2</summary>

forward: out = A_hat @ self.W(H), 한 줄 if 로 relu

</details>

<details><summary>힌트 3</summary>

ScratchGCN: conv1(in, hidden), conv2(hidden, classes)

</details>

<details><summary>힌트 4</summary>

forward: h 를 만들고 return_embeddings 면 h, 아니면 conv2(..., activation=False)

</details>

원본: PracticeCode_2.ipynb 셀 56


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# GCNLayer(nn.Module)
# 1. __init__(self, in_dim, out_dim): 부모 준비, self.W = bias 없는 nn.Linear(in_dim, out_dim)
# 2. forward(self, H, A_hat, activation=True):
#    out = A_hat 과 self.W(H) 의 행렬곱, activation 이면 torch.relu(out), 아니면 out 을 돌려줘요
class GCNLayer(nn.Module):
    ...


# ScratchGCN(nn.Module)
# 1. __init__(self, in_dim, hidden_dim, num_classes): 부모 준비,
#    self.conv1 = GCNLayer(in_dim, hidden_dim), self.conv2 = GCNLayer(hidden_dim, num_classes)
# 2. forward(self, x, A_hat, return_embeddings=False):
#    h = conv1(x, A_hat, activation=True), return_embeddings 면 h 를 돌려줘요
#    아니면 conv2(h, A_hat, activation=False) 를 돌려줘요
class ScratchGCN(nn.Module):
    ...


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F

class _RGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out

class _RScratchGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = _RGCNLayer(in_dim, hidden_dim)
        self.conv2 = _RGCNLayer(hidden_dim, num_classes)

    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)

def _r_graph():
    _e = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
    _A = torch.zeros(8, 8)
    for _u, _v in _e:
        _A[_u, _v] = 1.0
        _A[_v, _u] = 1.0
    _At = _A + torch.eye(8)
    _Dm = torch.diag(_At.sum(dim=1).pow(-0.5))
    _m = torch.zeros(8, dtype=torch.bool)
    _m[0] = True
    _m[7] = True
    return _A, _Dm @ _At @ _Dm, torch.eye(8), torch.tensor([0, 0, 0, 0, 1, 1, 1, 1]), _m

def _r_train(model, x, support, y, train_mask, epochs, lr):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    hl, ht, hf = [], [], []
    for _ep in range(1, epochs + 1):
        model.train()
        opt.zero_grad()
        lg = model(x, support)
        l = F.cross_entropy(lg[train_mask], y[train_mask])
        l.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            p = model(x, support).argmax(dim=1)
            ht.append((p[train_mask] == y[train_mask]).float().mean().item())
            hf.append((p == y).float().mean().item())
        hl.append(l.item())
    return hl, ht, hf

_A8, _AH8, _X8, _Y8, _M8 = _r_graph()
assert 'GCNLayer' in globals(), "GCNLayer 클래스를 만들어야 해요"
try:
    torch.manual_seed(0)
    _layer = GCNLayer(3, 2)
except Exception as _e:
    raise AssertionError(f"GCNLayer(3, 2) 로 층을 만들다가 에러가 났어요: {_e}. __init__(self, in_dim, out_dim) 과 super().__init__() 을 확인해요")
assert isinstance(_layer, nn.Module), "GCNLayer 은 nn.Module 을 물려받아야 해요: class GCNLayer(nn.Module):"
assert hasattr(_layer, 'W') and isinstance(_layer.W, nn.Linear), "self.W = nn.Linear(in_dim, out_dim, bias=False) 가 있어야 해요"
assert tuple(_layer.W.weight.shape) == (2, 3), f"W 의 weight 모양은 (출력 2, 입력 3) 이어야 해요. 지금은 {tuple(_layer.W.weight.shape)}"
assert _layer.W.bias is None, "원본은 bias=False 예요. 더하는 숫자(bias)가 없어야 해요"
torch.manual_seed(1)
_H = torch.randn(4, 3)
_Ah = torch.rand(4, 4)
try:
    _o1 = _layer(_H, _Ah)
    _o2 = _layer(_H, _Ah, activation=False)
except Exception as _e:
    raise AssertionError(f"층을 부르다가 에러가 났어요: {_e}. forward(self, H, A_hat, activation=True) 를 확인해요")
_raw = _Ah @ (_H @ _layer.W.weight.T)
assert _o2 is not None and torch.allclose(_o2, _raw, atol=1e-5), "activation=False 면 A_hat @ self.W(H) 그대로 돌려줘야 해요"
assert torch.allclose(_o1, torch.relu(_raw), atol=1e-5), "activation=True 면 torch.relu 로 음수를 0 으로 바꿔야 해요"
assert 'ScratchGCN' in globals(), "ScratchGCN 클래스를 만들어야 해요"
for _i, _h, _c in [(8, 16, 2), (5, 4, 3)]:
    try:
        torch.manual_seed(42)
        _m = ScratchGCN(_i, _h, _c)
    except Exception as _e:
        raise AssertionError(f"ScratchGCN({_i}, {_h}, {_c}) 를 만들다가 에러가 났어요: {_e}")
    torch.manual_seed(42)
    _r = _RScratchGCN(_i, _h, _c)
    assert hasattr(_m, 'conv1') and hasattr(_m, 'conv2'), "self.conv1, self.conv2 두 층이 있어야 해요"
    assert tuple(_m.conv1.W.weight.shape) == (_h, _i), f"conv1 은 GCNLayer(in_dim, hidden_dim) 이어야 해요. weight 모양이 {tuple(_m.conv1.W.weight.shape)} 예요"
    assert tuple(_m.conv2.W.weight.shape) == (_c, _h), f"conv2 는 GCNLayer(hidden_dim, num_classes) 이어야 해요. weight 모양이 {tuple(_m.conv2.W.weight.shape)} 예요"
    torch.manual_seed(3)
    _xx = torch.randn(_i, _i) if _i != 8 else _X8
    _aa = torch.rand(_i, _i) if _i != 8 else _AH8
    try:
        _out = _m(_xx, _aa)
        _emb = _m(_xx, _aa, return_embeddings=True)
    except Exception as _e:
        raise AssertionError(f"모델을 부르다가 에러가 났어요: {_e}. 층의 입력, 출력 칸 수를 확인해요")
    assert _out is not None and tuple(_out.shape) == (_i, _c), f"model(x, A_hat) 는 (노드 수 {_i}, 반 수 {_c}) 모양이어야 해요"
    assert _emb is not None and tuple(_emb.shape) == (_i, _h), "return_embeddings=True 면 첫 층 결과 h 를 돌려줘야 해요"
    assert torch.allclose(_out, _r(_xx, _aa), atol=1e-5), "마지막 층 결과가 원본과 달라요. 마지막 층은 activation=False 예요"
    assert torch.allclose(_emb, _r(_xx, _aa, return_embeddings=True), atol=1e-5), "숨은 표현 h 가 원본과 달라요. 첫 층은 activation=True 예요"

print("통과! 이제 2주차 셀 56 을 빈 칸에서 혼자 쓸 수 있어요.")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn
import torch.nn.functional as F

class GCNLayer(nn.Module):
    '''One layer: H' = activation(A_hat H W).'''

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out


class ScratchGCN(nn.Module):
    '''Two-layer GCN.'''

    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = GCNLayer(in_dim, hidden_dim)
        self.conv2 = GCNLayer(hidden_dim, num_classes)

    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)

```

</details>

## 11. train_dense_model 직접 짜기  (직접 짜기)

주석 단계만 보고 학습 함수 train_dense_model 을 짜요. 모델, 클래스, 데이터는 이미 들어 있어요.

- 07 에서 빈칸으로 채운 함수를 이번엔 통째로 써요. print 줄은 넣어도 되고 빼도 채점에는 상관없어요.
- 채점은 두 가지예요: 가짜 그래프에서 60바퀴 기록이 원본과 같은지, 다른 모델(층 하나짜리)과 다른 lr 에서도 같은지.
- 옵티마이저와 기록 리스트는 반복 **밖**에서 한 번만 만들어요. 안에서 만들면 매 바퀴 새로 시작해요.

<details><summary>생각 순서 보기 (힌트보다 먼저)</summary>

1. 입력과 출력부터 적어요. 모델, x, support, y, train_mask, epochs, lr 을 받아 리스트 세 개(손실, 훈련 정확도, 전체 정확도) 를 돌려줘요.
2. 반복 밖에 둘 것: `Adam(..., weight_decay=5e-4)` 옵티마이저와 빈 리스트 세 개. 안에서 만들면 매 바퀴 처음부터 다시 시작해요.
3. 반복 안은 두 덩어리예요. 앞은 학습(train 모드, zero_grad, 예측, 손실, backward, step), 뒤는 채점(eval 모드, no_grad, 다시 예측, 정확도)이에요.
4. 조심할 곳: 정확도는 이미 있는 `accuracy_from_logits` 를 불러 쓰고, 리스트에는 텐서가 아니라 `loss.item()` 숫자를 붙여요.
5. 확인: `len(gcn_losses)` 가 60 이고 마지막 손실이 첫 손실보다 작으면 맞아요.

슈도코드

```text
Adam 옵티마이저를 lr 과 weight_decay 5e-4 로 만든다
빈 리스트 history_loss, history_train, history_full 을 만든다
epoch 를 1 부터 epochs 까지 돌며
    model 을 학습 모드로 두고 기울기를 지운다
    logits 를 얻고 train_mask 노드만으로 손실을 구한다
    backward 로 기울기를 구하고 step 으로 가중치를 고친다
    model 을 평가 모드로 두고 기울기 없이 다시 예측한다
    accuracy_from_logits 로 train_acc 와 full_acc 를 얻는다
    손실 숫자와 두 정확도를 세 리스트에 붙인다
세 리스트를 돌려준다
```

</details>

<details><summary>힌트 1</summary>

optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)

</details>

<details><summary>힌트 2</summary>

for epoch in range(1, epochs + 1): 안에서 train 여섯 줄

</details>

<details><summary>힌트 3</summary>

eval, no_grad 안에서 accuracy_from_logits

</details>

<details><summary>힌트 4</summary>

세 리스트에 append, 반복 뒤 return

</details>

원본: PracticeCode_2.ipynb 셀 49


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GCNLayer(nn.Module):
    '''One layer: H' = activation(A_hat H W).'''

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out


class ScratchGCN(nn.Module):
    '''Two-layer GCN.'''

    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = GCNLayer(in_dim, hidden_dim)
        self.conv2 = GCNLayer(hidden_dim, num_classes)

    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)


def accuracy_from_logits(logits, y, train_mask):
    pred = logits.argmax(dim=1)
    train_acc = (pred[train_mask] == y[train_mask]).float().mean().item()
    full_acc = (pred == y).float().mean().item()
    return pred, train_acc, full_acc


# train_dense_model(model, x, support, y, train_mask, epochs=300, lr=0.01, name="model")
# 1. optimizer = Adam(model.parameters(), lr=lr, weight_decay=5e-4)
# 2. history_loss, history_train, history_full = 빈 리스트 세 개
# 3. epoch 를 1 부터 epochs 까지 돌면서
#    3-1. model.train(), optimizer.zero_grad()
#    3-2. logits = model(x, support)
#    3-3. loss = train_mask 노드만 F.cross_entropy, loss.backward(), optimizer.step()
#    3-4. model.eval() 하고 with torch.no_grad(): 안에서
#         logits_eval = model(x, support), accuracy_from_logits 로 train_acc, full_acc
#    3-5. 세 리스트에 loss.item(), train_acc, full_acc 를 붙여요
# 4. 세 리스트를 돌려줘요
def train_dense_model(model, x, support, y, train_mask, epochs=300, lr=0.01, name="model"):
    ...


edges = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
num_nodes = 8
A = torch.zeros(num_nodes, num_nodes)
for u, v in edges:
    A[u, v] = 1.0
    A[v, u] = 1.0
A_tilde = A + torch.eye(num_nodes)
deg = A_tilde.sum(dim=1)
D_inv_sqrt = torch.diag(deg.pow(-0.5))
A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt
x = torch.eye(num_nodes)
y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1])
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
train_mask[0] = True
train_mask[7] = True

torch.manual_seed(42)
scratch_gcn = ScratchGCN(x.size(1), hidden_dim=16, num_classes=2)
gcn_losses, gcn_train_accs, gcn_full_accs = train_dense_model(
    scratch_gcn, x, A_hat, y, train_mask, epochs=60, name="the scratch GCN"
)


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F

class _RGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out

class _RScratchGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = _RGCNLayer(in_dim, hidden_dim)
        self.conv2 = _RGCNLayer(hidden_dim, num_classes)

    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)

def _r_graph():
    _e = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
    _A = torch.zeros(8, 8)
    for _u, _v in _e:
        _A[_u, _v] = 1.0
        _A[_v, _u] = 1.0
    _At = _A + torch.eye(8)
    _Dm = torch.diag(_At.sum(dim=1).pow(-0.5))
    _m = torch.zeros(8, dtype=torch.bool)
    _m[0] = True
    _m[7] = True
    return _A, _Dm @ _At @ _Dm, torch.eye(8), torch.tensor([0, 0, 0, 0, 1, 1, 1, 1]), _m

def _r_train(model, x, support, y, train_mask, epochs, lr):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    hl, ht, hf = [], [], []
    for _ep in range(1, epochs + 1):
        model.train()
        opt.zero_grad()
        lg = model(x, support)
        l = F.cross_entropy(lg[train_mask], y[train_mask])
        l.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            p = model(x, support).argmax(dim=1)
            ht.append((p[train_mask] == y[train_mask]).float().mean().item())
            hf.append((p == y).float().mean().item())
        hl.append(l.item())
    return hl, ht, hf

_A8, _AH8, _X8, _Y8, _M8 = _r_graph()
assert 'gcn_losses' in globals() and len(gcn_losses) == 60, "gcn_losses 에 60바퀴 손실이 모여야 해요"
torch.manual_seed(42)
_rm = _RScratchGCN(8, 16, 2)
_rl, _rt, _rf = _r_train(_rm, _X8, _AH8, _Y8, _M8, 60, 0.01)
assert abs(gcn_losses[0] - _rl[0]) < 1e-5, f"첫 손실이 원본 {_rl[0]:.4f} 와 달라요(지금 {gcn_losses[0]:.4f}). train_mask 인 노드만 채점했는지 봐요"
assert max(abs(a - b) for a, b in zip(gcn_losses, _rl)) < 1e-4, "손실 기록이 원본과 달라요. zero_grad, backward, step 순서를 확인해요"
assert gcn_full_accs == _rf, "전체 정확도 기록이 원본과 달라요. model.eval() 과 torch.no_grad() 안에서 다시 계산했나요?"
assert gcn_losses[-1] < gcn_losses[0], "학습하면 손실이 줄어야 해요"
torch.manual_seed(9)
_s1 = nn.Linear(8, 2)
torch.manual_seed(9)
_s2 = nn.Linear(8, 2)
class _Wrap(nn.Module):
    def __init__(self, lin):
        super().__init__()
        self.lin = lin
    def forward(self, x, support):
        return self.lin(support @ x)
_g = train_dense_model(_Wrap(_s1), _X8, _AH8, _Y8, _M8, epochs=15, lr=0.05, name="t")
_w = _r_train(_Wrap(_s2), _X8, _AH8, _Y8, _M8, 15, 0.05)
assert _g is not None and len(_g) == 3, "history_loss, history_train, history_full 세 리스트를 돌려줘야 해요"
assert len(_g[0]) == 15, "epochs=15 면 기록이 15개여야 해요. range(1, epochs + 1) 을 확인해요"
assert max(abs(a - b) for a, b in zip(_g[0], _w[0])) < 1e-5, "다른 모델, lr=0.05 에서 손실이 원본과 달라요. lr=lr 과 weight_decay=5e-4 를 확인해요"
assert _g[1] == _w[1], "훈련 정확도 기록이 원본과 달라요"

print("통과! 원본 셀 49 를 혼자 썼어요. w2-pyg 의 train_sparse_model 도 거의 같은 모양이에요.")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn
import torch.nn.functional as F

class GCNLayer(nn.Module):
    '''One layer: H' = activation(A_hat H W).'''

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out


class ScratchGCN(nn.Module):
    '''Two-layer GCN.'''

    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = GCNLayer(in_dim, hidden_dim)
        self.conv2 = GCNLayer(hidden_dim, num_classes)

    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)


def accuracy_from_logits(logits, y, train_mask):
    pred = logits.argmax(dim=1)
    train_acc = (pred[train_mask] == y[train_mask]).float().mean().item()
    full_acc = (pred == y).float().mean().item()
    return pred, train_acc, full_acc


def train_dense_model(model, x, support, y, train_mask, epochs=300, lr=0.01, name="model"):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    history_loss, history_train, history_full = [], [], []

    print(f"Training {name}")
    print("=" * 60)
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(x, support)
        loss = F.cross_entropy(logits[train_mask], y[train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            logits_eval = model(x, support)
            _, train_acc, full_acc = accuracy_from_logits(logits_eval, y, train_mask)

        history_loss.append(loss.item())
        history_train.append(train_acc)
        history_full.append(full_acc)

        if epoch % 20 == 0 or epoch == 1:
            print(
                f"Epoch {epoch:03d}/{epochs} | loss={loss.item():.4f} "
                f"| train acc={train_acc:.4f} | full-graph acc={full_acc:.4f}"
            )

    print("=" * 60)
    return history_loss, history_train, history_full


edges = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
num_nodes = 8
A = torch.zeros(num_nodes, num_nodes)
for u, v in edges:
    A[u, v] = 1.0
    A[v, u] = 1.0
A_tilde = A + torch.eye(num_nodes)
deg = A_tilde.sum(dim=1)
D_inv_sqrt = torch.diag(deg.pow(-0.5))
A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt
x = torch.eye(num_nodes)
y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1])
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
train_mask[0] = True
train_mask[7] = True

torch.manual_seed(42)
scratch_gcn = ScratchGCN(x.size(1), hidden_dim=16, num_classes=2)
gcn_losses, gcn_train_accs, gcn_full_accs = train_dense_model(
    scratch_gcn, x, A_hat, y, train_mask, epochs=60, name="the scratch GCN"
)

```

</details>

## 12. 학습하고 예측까지 한 번에  (직접 짜기)

ScratchGCN 을 만들어 100바퀴 학습하고, 학습이 끝난 모델로 숨은 표현(gcn_embeddings)과 예측 반(gcn_pred)을 numpy 배열로 꺼내요.

- 원본은 PyG 의 가라테 클럽(34명)을 쓰지만, 채점을 빠르게 하려고 8명짜리 가짜 그래프(0~3번 한 무리, 4~7번 한 무리, 3번과 4번이 다리)로 바꿨어요. 내려받는 데이터는 없어요.
- 원본 셀 56 의 학습 세 줄과 셀 57 의 예측 블록을 합친 과제예요. 클래스와 학습 함수는 이미 들어 있어요.
- `.cpu().numpy()` 는 텐서를 numpy 배열로 바꿔요. t-SNE 같은 sklearn 도구는 numpy 를 받아요 (c09).
- seed 42 로 모델을 만든 **바로 다음에** 학습해야 원본과 숫자가 같아요.
- 8명 가짜 그래프에서는 0, 7번 두 명만 답을 알려 줘도 전원을 맞혀요(정확도 1.0).

<details><summary>생각 순서 보기 (힌트보다 먼저)</summary>

1. 만들어야 할 변수부터 적어요. `gcn_losses` (100개), `gcn_embeddings` (8, 16) numpy 배열, `gcn_pred` (8,) numpy 배열이에요.
2. 순서가 중요해요. seed 42 를 정하고, 모델을 만들고, 바로 학습해요. 사이에 다른 무작위 코드가 끼면 숫자가 달라져요.
3. 학습이 끝나면 채점 모드예요. `scratch_gcn.eval()` 과 `with torch.no_grad():` 안에서 모델을 두 번 불러요. 한 번은 `return_embeddings=True`, 한 번은 점수예요.
4. 조심할 곳: 텐서를 numpy 로 바꿀 때 `.cpu().numpy()`, 예측은 줄마다 골라야 해서 `argmax(dim=1)` 이에요.
5. 확인: `gcn_pred.tolist()` 가 [0, 0, 0, 0, 1, 1, 1, 1] 이고 마지막 전체 정확도가 1.0 이면 맞아요.

슈도코드

```text
seed 를 42 로 정한다
입력 칸 수, 16, 2 로 ScratchGCN 을 만들어 scratch_gcn 에 둔다
train_dense_model 로 100바퀴 학습하고 기록 세 개를 받는다
scratch_gcn 을 평가 모드로 바꾼다
기울기 계산 없이
    숨은 표현을 꺼내 numpy 로 바꿔 gcn_embeddings 에 둔다
    반별 점수를 gcn_logits 에 둔다
    줄마다 가장 큰 칸 번호를 numpy 로 바꿔 gcn_pred 에 둔다
```

</details>

<details><summary>힌트 1</summary>

seed 42, ScratchGCN(x.size(1), hidden_dim=16, num_classes=2)

</details>

<details><summary>힌트 2</summary>

train_dense_model(..., epochs=100, ...)

</details>

<details><summary>힌트 3</summary>

scratch_gcn.eval() 과 with torch.no_grad():

</details>

<details><summary>힌트 4</summary>

embeddings 는 return_embeddings=True, pred 는 argmax(dim=1), 둘 다 .cpu().numpy()

</details>

원본: PracticeCode_2.ipynb 셀 56, 57, 70


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GCNLayer(nn.Module):
    '''One layer: H' = activation(A_hat H W).'''

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out


class ScratchGCN(nn.Module):
    '''Two-layer GCN.'''

    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = GCNLayer(in_dim, hidden_dim)
        self.conv2 = GCNLayer(hidden_dim, num_classes)

    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)


def accuracy_from_logits(logits, y, train_mask):
    pred = logits.argmax(dim=1)
    train_acc = (pred[train_mask] == y[train_mask]).float().mean().item()
    full_acc = (pred == y).float().mean().item()
    return pred, train_acc, full_acc


def train_dense_model(model, x, support, y, train_mask, epochs=300, lr=0.01, name="model"):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    history_loss, history_train, history_full = [], [], []

    print(f"Training {name}")
    print("=" * 60)
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(x, support)
        loss = F.cross_entropy(logits[train_mask], y[train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            logits_eval = model(x, support)
            _, train_acc, full_acc = accuracy_from_logits(logits_eval, y, train_mask)

        history_loss.append(loss.item())
        history_train.append(train_acc)
        history_full.append(full_acc)

        if epoch % 20 == 0 or epoch == 1:
            print(
                f"Epoch {epoch:03d}/{epochs} | loss={loss.item():.4f} "
                f"| train acc={train_acc:.4f} | full-graph acc={full_acc:.4f}"
            )

    print("=" * 60)
    return history_loss, history_train, history_full


edges = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
num_nodes = 8
A = torch.zeros(num_nodes, num_nodes)
for u, v in edges:
    A[u, v] = 1.0
    A[v, u] = 1.0
A_tilde = A + torch.eye(num_nodes)
deg = A_tilde.sum(dim=1)
D_inv_sqrt = torch.diag(deg.pow(-0.5))
A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt
x = torch.eye(num_nodes)
y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1])
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
train_mask[0] = True
train_mask[7] = True

# 1. torch.manual_seed(42) 다음 scratch_gcn = ScratchGCN(입력 칸 수, hidden_dim=16, num_classes=2)
# 2. gcn_losses, gcn_train_accs, gcn_full_accs = train_dense_model(..., epochs=100, name="the scratch GCN")
# 3. scratch_gcn 을 평가 모드로 바꾸고, 기울기 계산 없이:
#    3-1. gcn_embeddings = 숨은 표현을 numpy 배열로
#    3-2. gcn_logits = 반별 점수
#    3-3. gcn_pred = 줄마다 가장 큰 칸 번호를 numpy 배열로


In [ ]:
# 채점 칸: 위 칸을 실행한 다음 이 칸을 실행하세요
import torch
import torch.nn as nn
import torch.nn.functional as F

class _RGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out

class _RScratchGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = _RGCNLayer(in_dim, hidden_dim)
        self.conv2 = _RGCNLayer(hidden_dim, num_classes)

    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)

def _r_graph():
    _e = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
    _A = torch.zeros(8, 8)
    for _u, _v in _e:
        _A[_u, _v] = 1.0
        _A[_v, _u] = 1.0
    _At = _A + torch.eye(8)
    _Dm = torch.diag(_At.sum(dim=1).pow(-0.5))
    _m = torch.zeros(8, dtype=torch.bool)
    _m[0] = True
    _m[7] = True
    return _A, _Dm @ _At @ _Dm, torch.eye(8), torch.tensor([0, 0, 0, 0, 1, 1, 1, 1]), _m

def _r_train(model, x, support, y, train_mask, epochs, lr):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    hl, ht, hf = [], [], []
    for _ep in range(1, epochs + 1):
        model.train()
        opt.zero_grad()
        lg = model(x, support)
        l = F.cross_entropy(lg[train_mask], y[train_mask])
        l.backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            p = model(x, support).argmax(dim=1)
            ht.append((p[train_mask] == y[train_mask]).float().mean().item())
            hf.append((p == y).float().mean().item())
        hl.append(l.item())
    return hl, ht, hf

_A8, _AH8, _X8, _Y8, _M8 = _r_graph()
import numpy as np
for _n in ['scratch_gcn', 'gcn_losses', 'gcn_embeddings', 'gcn_pred']:
    assert _n in globals(), f"{_n} 를 만들어야 해요"
torch.manual_seed(42)
_rm = _RScratchGCN(8, 16, 2)
_rl, _rt, _rf = _r_train(_rm, _X8, _AH8, _Y8, _M8, 100, 0.01)
assert len(gcn_losses) == 100, f"100바퀴 학습해야 해요. 지금 기록은 {len(gcn_losses)}개예요"
assert abs(gcn_losses[-1] - _rl[-1]) < 1e-4, f"마지막 손실이 원본 {_rl[-1]:.4f} 와 달라요. seed 42 로 ScratchGCN(8, 16, 2) 를 만들고 바로 학습했나요?"
_rm.eval()
with torch.no_grad():
    _re = _rm(_X8, _AH8, return_embeddings=True).numpy()
    _rp = _rm(_X8, _AH8).argmax(dim=1).numpy()
assert isinstance(gcn_embeddings, np.ndarray) and isinstance(gcn_pred, np.ndarray), ".cpu().numpy() 로 numpy 배열로 바꿔요"
assert gcn_embeddings.shape == (8, 16), f"gcn_embeddings 는 (8, 16) 이어야 해요. return_embeddings=True 를 확인해요. 지금은 {gcn_embeddings.shape}"
assert np.allclose(gcn_embeddings, _re, atol=1e-4), "숨은 표현이 원본과 달라요. scratch_gcn.eval() 과 torch.no_grad() 안에서 꺼냈나요?"
assert gcn_pred.tolist() == _rp.tolist(), f"예측이 원본 {_rp.tolist()} 와 달라요. argmax(dim=1) 을 확인해요"

print("통과! 노트북 셀 57 은 이 gcn_embeddings 를 t-SNE 로 그려서 두 무리가 갈라지는 걸 보여 줘요.")

<details><summary>정답 보기</summary>

```python
import torch
import torch.nn as nn
import torch.nn.functional as F

class GCNLayer(nn.Module):
    '''One layer: H' = activation(A_hat H W).'''

    def __init__(self, in_dim, out_dim):
        super().__init__()
        self.W = nn.Linear(in_dim, out_dim, bias=False)

    def forward(self, H, A_hat, activation=True):
        out = A_hat @ self.W(H)
        return torch.relu(out) if activation else out


class ScratchGCN(nn.Module):
    '''Two-layer GCN.'''

    def __init__(self, in_dim, hidden_dim, num_classes):
        super().__init__()
        self.conv1 = GCNLayer(in_dim, hidden_dim)
        self.conv2 = GCNLayer(hidden_dim, num_classes)

    def forward(self, x, A_hat, return_embeddings=False):
        h = self.conv1(x, A_hat, activation=True)
        if return_embeddings:
            return h
        return self.conv2(h, A_hat, activation=False)


def accuracy_from_logits(logits, y, train_mask):
    pred = logits.argmax(dim=1)
    train_acc = (pred[train_mask] == y[train_mask]).float().mean().item()
    full_acc = (pred == y).float().mean().item()
    return pred, train_acc, full_acc


def train_dense_model(model, x, support, y, train_mask, epochs=300, lr=0.01, name="model"):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    history_loss, history_train, history_full = [], [], []

    print(f"Training {name}")
    print("=" * 60)
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(x, support)
        loss = F.cross_entropy(logits[train_mask], y[train_mask])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            logits_eval = model(x, support)
            _, train_acc, full_acc = accuracy_from_logits(logits_eval, y, train_mask)

        history_loss.append(loss.item())
        history_train.append(train_acc)
        history_full.append(full_acc)

        if epoch % 20 == 0 or epoch == 1:
            print(
                f"Epoch {epoch:03d}/{epochs} | loss={loss.item():.4f} "
                f"| train acc={train_acc:.4f} | full-graph acc={full_acc:.4f}"
            )

    print("=" * 60)
    return history_loss, history_train, history_full


edges = [(0, 1), (0, 2), (1, 2), (0, 3), (2, 3), (3, 4), (4, 5), (4, 6), (5, 6), (5, 7), (6, 7)]
num_nodes = 8
A = torch.zeros(num_nodes, num_nodes)
for u, v in edges:
    A[u, v] = 1.0
    A[v, u] = 1.0
A_tilde = A + torch.eye(num_nodes)
deg = A_tilde.sum(dim=1)
D_inv_sqrt = torch.diag(deg.pow(-0.5))
A_hat = D_inv_sqrt @ A_tilde @ D_inv_sqrt
x = torch.eye(num_nodes)
y = torch.tensor([0, 0, 0, 0, 1, 1, 1, 1])
train_mask = torch.zeros(num_nodes, dtype=torch.bool)
train_mask[0] = True
train_mask[7] = True

torch.manual_seed(42)
scratch_gcn = ScratchGCN(x.size(1), hidden_dim=16, num_classes=2)
gcn_losses, gcn_train_accs, gcn_full_accs = train_dense_model(
    scratch_gcn, x, A_hat, y, train_mask, epochs=100, name="the scratch GCN"
)

scratch_gcn.eval()
with torch.no_grad():
    gcn_embeddings = scratch_gcn(x, A_hat, return_embeddings=True).cpu().numpy()
    gcn_logits = scratch_gcn(x, A_hat)
    gcn_pred = gcn_logits.argmax(dim=1).cpu().numpy()

print("pred:", gcn_pred.tolist())
print("final full-graph acc:", gcn_full_accs[-1])

```

</details>